# Tahap 6: Generate & Export Visual Assets untuk Laporan PDF
**Silabus:** Jiawei Han - Data Mining: Concepts and Techniques, Ch. 3 (Before vs. After Deliverable)  
**DataFrame Target:** `df_raw` (Mentah) vs `df_clean` (Data Hasil Preprocessing)  
**Fokus:**  
1. Export 4 Gambar Visualisasi Beresolusi Tinggi (DPI 300) ke folder `figures/`:  
   - **Figure 1**: `fig1_before_after_cleaning.png` (Missing Values & Distribusi ADR Boxplot).  
   - **Figure 2**: `fig2_data_integration.png` (Chi-Square Contingency & Pearson Correlation Heatmap).  
   - **Figure 3**: `fig3_data_reduction_pca.png` (Scree Plot Variance & Proyeksi Subruang 2D PCA).  
   - **Figure 4**: `fig4_discretization_binning.png` (Equal-Width vs Equal-Frequency Lead Time Binning).  
2. Export **Tabel Komparasi Kuantitatif Final (Markdown)** untuk langsung disalin ke naskah laporan PDF.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from IPython.display import display, Markdown

# 0. Buat folder figures/ jika belum ada
os.makedirs('figures', exist_ok=True)

# Memuat Data Mentah & Data Preprocessed
csv_file = "hotel_booking_data.csv"
df_raw = pd.read_csv(csv_file)

cols_to_drop = ['name', 'email', 'phone-number', 'credit_card', 'reservation_status', 'reservation_status_date']
df_clean = df_raw.drop(columns=[c for c in cols_to_drop if c in df_raw.columns]).drop_duplicates()

adr_anom = (df_clean['adr'] < 0) | (df_clean['adr'] > 1000)
total_guests = df_clean['adults'] + df_clean['children'].fillna(0) + df_clean['babies'].fillna(0)
df_clean = df_clean[~adr_anom & (total_guests > 0)].copy()

df_clean['children'] = df_clean['children'].fillna(0).astype(int)
df_clean['country'] = df_clean['country'].fillna('Unknown')
df_clean['agent'] = df_clean['agent'].fillna(0)
df_clean['company'] = df_clean['company'].fillna(0)

# Feature Engineering
df_clean['total_stay_nights'] = df_clean['stays_in_weekend_nights'] + df_clean['stays_in_week_nights']
df_clean['is_room_changed'] = (df_clean['reserved_room_type'] != df_clean['assigned_room_type']).astype(int)

# Discretization
labels = ['Very Short', 'Short', 'Medium', 'Long']
df_clean['lead_time_category'] = pd.qcut(df_clean['lead_time'], q=4, labels=labels)

print("Dataset df_raw & df_clean siap diproses untuk pembuatan visual assets!\n")


# =========================================================
# 1. FIGURE 1: DATA CLEANING BEFORE VS AFTER
# =========================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot A: Barplot Missing Values
raw_missing = df_raw[['company', 'agent', 'country', 'children']].isnull().sum()
clean_missing = df_clean[['company', 'agent', 'country', 'children']].isnull().sum()
missing_df = pd.DataFrame({'Before (df_raw)': raw_missing, 'After (df_clean)': clean_missing})
missing_df.plot(kind='bar', ax=axes[0], color=['#e74c3c', '#2ecc71'], alpha=0.85)
axes[0].set_title('Subplot A: Perbandingan Missing Values per Kolom', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Jumlah Missing Values', fontweight='bold')
axes[0].set_xticklabels(missing_df.index, rotation=0)

# Subplot B: Boxplot Distribusi ADR
sns.boxplot(data=[df_raw['adr'], df_clean['adr']], ax=axes[1], palette=['#e74c3c', '#2ecc71'])
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['Before (df_raw)', 'After (df_clean)'])
axes[1].set_title('Subplot B: Distribusi ADR (Eliminasi ADR < 0 & > 1000)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Average Daily Rate (ADR)', fontweight='bold')

plt.suptitle('Figure 1: Data Cleaning (Before vs. After)', fontsize=13, fontweight='bold')
plt.tight_layout()
fig1_path = 'figures/fig1_before_after_cleaning.png'
plt.savefig(fig1_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"▶ Figure 1 berhasil diekspor ke '{fig1_path}' (DPI=300)")


# =========================================================
# 2. FIGURE 2: DATA INTEGRATION & REDUNDANCY
# =========================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Subplot A: Contingency Heatmap Chi-Square
obs = pd.crosstab(df_clean['reserved_room_type'], df_clean['assigned_room_type'])
sns.heatmap(obs, ax=axes[0], annot=True, fmt='d', cmap='YlGnBu', cbar=True)
axes[0].set_title('Subplot A: Contingency Heatmap (Reserved vs Assigned)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Assigned Room Type', fontweight='bold')
axes[0].set_ylabel('Reserved Room Type', fontweight='bold')

# Subplot B: Pearson Correlation Matrix
num_cols = ['lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'booking_changes', 'adr', 'total_of_special_requests']
corr = df_clean[num_cols].corr()
sns.heatmap(corr, ax=axes[1], annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
axes[1].set_title('Subplot B: Pearson Correlation Matrix', fontsize=11, fontweight='bold')

plt.suptitle('Figure 2: Data Integration & Redundancy Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
fig2_path = 'figures/fig2_data_integration.png'
plt.savefig(fig2_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"▶ Figure 2 berhasil diekspor ke '{fig2_path}' (DPI=300)")


# =========================================================
# 3. FIGURE 3: DATA REDUCTION (PCA ANALYSIS)
# =========================================================
pca_cols = ['lead_time', 'total_stay_nights', 'adults', 'children', 'booking_changes', 'adr', 'days_in_waiting_list', 'total_of_special_requests']
X_scaled = StandardScaler().fit_transform(df_clean[pca_cols])
pca = PCA()
X_pca = pca.fit_transform(X_scaled)
cum_evr = np.cumsum(pca.explained_variance_ratio_) * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Subplot A: Scree Plot Cumulative EVR
pcs = [f'PC{i+1}' for i in range(len(pca_cols))]
axes[0].plot(pcs, cum_evr, color='red', marker='o', linewidth=2, label='Cumulative Variance (%)')
axes[0].axhline(y=70, color='black', linestyle='--', label='Threshold Retention (70%)')
axes[0].set_title('Subplot A: Cumulative Explained Variance Ratio', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Komponen Utama (PCA)', fontweight='bold')
axes[0].set_ylabel('Cumulative Variance (%)', fontweight='bold')
axes[0].legend()

# Subplot B: 2D Subspace Scatter Plot (PC1 vs PC2)
df_pca = pd.DataFrame({'PC1': X_pca[:, 0], 'PC2': X_pca[:, 1], 'is_canceled': df_clean['is_canceled'].values})
sns.scatterplot(data=df_pca, x='PC1', y='PC2', hue='is_canceled', palette={0: '#2ecc71', 1: '#e74c3c'}, alpha=0.5, s=15, ax=axes[1])
axes[1].set_title('Subplot B: Proyeksi Subruang 2D (PC1 vs PC2)', fontsize=11, fontweight='bold')
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.2f}% Varians)", fontweight='bold')
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.2f}% Varians)", fontweight='bold')

plt.suptitle('Figure 3: Data Reduction (PCA Analysis)', fontsize=13, fontweight='bold')
plt.tight_layout()
fig3_path = 'figures/fig3_data_reduction_pca.png'
plt.savefig(fig3_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"▶ Figure 3 berhasil diekspor ke '{fig3_path}' (DPI=300)")


# =========================================================
# 4. FIGURE 4: DISCRETIZATION & BINNING ANALYSIS
# =========================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_clean['eq_width'] = pd.cut(df_clean['lead_time'], bins=4)
w_counts = df_clean['eq_width'].value_counts().sort_index().values
f_counts = df_clean['lead_time_category'].value_counts()[labels].values

# Subplot A: Equal-Width
axes[0].bar(range(4), w_counts, color='#e74c3c', alpha=0.85)
axes[0].set_xticks(range(4))
axes[0].set_xticklabels([f'Bin {i+1}' for i in range(4)])
axes[0].set_title('Subplot A: Equal-Width Partitioning (Miring/Skewed ke Bin 1)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Jumlah Record (Frekuensi)', fontweight='bold')
for i, v in enumerate(w_counts):
    axes[0].text(i, v + 1000, f"{v:,}", ha='center', fontweight='bold', fontsize=9)

# Subplot B: Equal-Frequency
axes[1].bar(labels, f_counts, color='#2ecc71', alpha=0.85)
axes[1].set_title('Subplot B: Equal-Frequency Partitioning (Seimbang ~25%)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Jumlah Record (Frekuensi)', fontweight='bold')
for i, v in enumerate(f_counts):
    axes[1].text(i, v + 500, f"{v:,}", ha='center', fontweight='bold', fontsize=9)

plt.suptitle('Figure 4: Data Transformation & Discretization (Lead Time Binning)', fontsize=13, fontweight='bold')
plt.tight_layout()
fig4_path = 'figures/fig4_discretization_binning.png'
plt.savefig(fig4_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"▶ Figure 4 berhasil diekspor ke '{fig4_path}' (DPI=300)")

df_clean.drop(columns=['eq_width'], inplace=True)


# =========================================================
# 5. TABEL KOMPARASI KUANTITATIF FINAL (MARKDOWN EXPORT)
# =========================================================
print("\n" + "=" * 80)
print("5. TABEL KOMPARASI KUANTITATIF FINAL (BEFORE VS AFTER)")
print("=" * 80)

raw_dup = df_raw.drop(columns=[c for c in cols_to_drop if c in df_raw.columns]).duplicated().sum()
raw_anom = ((df_raw['adr'] < 0) | (df_raw['adr'] > 1000)).sum()
tg_raw = df_raw['adults'] + df_raw['children'].fillna(0) + df_raw['babies'].fillna(0)
raw_ghost = (tg_raw == 0).sum()

comp_df = pd.DataFrame({
    'Metrik Kualitas Data': [
        'Total Baris (Records)',
        'Total Kolom (Attributes)',
        'Total Missing Cells',
        'Real Duplicate Rows',
        'Impossible/Extreme ADR Records',
        'Ghost Bookings (0 Guests)'
    ],
    'Before (df_raw)': [
        f'{len(df_raw):,}',
        f'{df_raw.shape[1]}',
        f'{df_raw.isnull().sum().sum():,}',
        f'{raw_dup:,}',
        f'{raw_anom:,}',
        f'{raw_ghost:,}'
    ],
    'After (df_clean)': [
        f'{len(df_clean):,}',
        f'{df_clean.shape[1]}',
        '0',
        '0',
        '0',
        '0'
    ]
})

print("\n--- Tabel Format Markdown (Siap Disalin ke Laporan PDF) ---")
markdown_table = comp_df.to_markdown(index=False)
print(markdown_table)

display(comp_df)

### Ringkasan Aset Visual & Tabel Komparasi untuk Laporan PDF

Seluruh gambar visualisasi hasil analisis telah diekspor ke folder `figures/` dengan spesifikasi cetak tinggi (**DPI 300, format PNG**):

1. **`fig1_before_after_cleaning.png`**:
   - Memvisualisasikan keberhasilan penanganan missing values (penurunan dari 129k+ sel kosong menjadi 0) dan eliminasi outlier ekstrim/negatif pada kolom ADR.
2. **`fig2_data_integration.png`**:
   - Menyajikan Heatmap Kontingensi Chi-Square (Reserved vs Assigned Room Type) serta Heatmap Korelasi Pearson untuk 8 atribut numerik.
3. **`fig3_data_reduction_pca.png`**:
   - Menyajikan Scree Plot varians akumulatif komponen utama PCA (threshold retention 70%) dan proyeksi subruang 2D (PC1 vs PC2) berdasarkan variabel target `is_canceled`.
4. **`fig4_discretization_binning.png`**:
   - Membandingkan kegagalan metode *Equal-Width* pada data *skewed* lawan keunggulan metode *Equal-Frequency* yang menghasilkan 4 kategori proporsional (~25% per bin).

Tabel perbandingan kuantitatif di atas dapat langsung disalin ke bab *Result & Discussion* laporan PDF.